In [ ]:
import os
os.environ["MAX_JOBS"] = "1"

import torch
import torch.nn.functional as F

from my_cnn_ext import my_cnn


## Convolution


In [ ]:
torch.manual_seed(0)

N, C_in, H, W = 4, 3, 32, 32
C_out = 8

x = torch.randn(N, C_in, H, W, device="cuda")
weight = torch.randn(C_out, C_in, 3, 3, device="cuda")

y_torch = F.conv2d(x, weight)
y_custom = my_cnn.my_convolution(x, weight)

print("torch :", tuple(y_torch.shape))
print("custom:", tuple(y_custom.shape))
print("max abs err:", (y_torch - y_custom).abs().max().item())

torch.testing.assert_close(y_custom, y_torch)
print("CONV OK")


## ReLU


In [ ]:
torch.manual_seed(0)

x = torch.randn(4, 3, 30, 30, device="cuda")
x[0, 0, 0, :5] = 0.0  # exact zeros must stay zero

y_torch = torch.relu(x)
y_custom = my_cnn.my_relu(x)

print("max abs err:", (y_torch - y_custom).abs().max().item())
torch.testing.assert_close(y_custom, y_torch)

# flat, not a multiple of the 256-thread block
flat = torch.randn(1000, device="cuda") - 0.5
torch.testing.assert_close(my_cnn.my_relu(flat), torch.relu(flat))
print("RELU OK")


In [ ]:
from layers.python.maxpool import CustomMaxPool2d

x = torch.randn(2, 3, 6, 6, device="cuda")
x_torch = x.detach().clone().requires_grad_()
x_custom = x.detach().clone().requires_grad_()
pool_torch = F.max_pool2d(x_torch, 2)
pool_custom = CustomMaxPool2d(2)(x_custom)
grad_output = torch.randn_like(pool_torch)
grad_torch = torch.autograd.grad((pool_torch * grad_output).sum(), x_torch)[0]
grad_custom = torch.autograd.grad(
    (pool_custom * grad_output).sum(), x_custom
)[0]

torch.testing.assert_close(pool_custom, pool_torch)
torch.testing.assert_close(grad_custom, grad_torch)
print("MAXPOOL OK")
